# 02 — Content-Based Filtering
Recommend items similar to what a user has interacted with, using item text features.

**Two approaches:**
1. TF-IDF + Cosine Similarity (fast, lightweight)
2. Sentence-BERT Embeddings (slower, semantically richer)

Run notebook 01 first to generate the processed data files.

## 1. Load Data

In [ ]:
import pandas as pd
import numpy as np

catalog = pd.read_csv('../data/processed/item_catalog.csv')
train   = pd.read_csv('../data/processed/train.csv')

print(f'Catalog size:       {len(catalog):,} items')
print(f'Train interactions: {len(train):,}')
catalog.head()

## 2. TF-IDF Similarity
Fast approach. Works well when item titles are distinct enough.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Build TF-IDF matrix from item titles
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),   # unigrams + bigrams
    min_df=2,             # ignore very rare terms
    stop_words='english'
)

tfidf_matrix = tfidf.fit_transform(catalog['item_title'])

print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')
print(f'Vocabulary size:     {len(tfidf.vocabulary_):,}')

In [ ]:
# Map item_id → index in catalog for fast lookups
item_to_idx = {item_id: idx for idx, item_id in enumerate(catalog['item_id'])}
idx_to_item = {idx: item_id for item_id, idx in item_to_idx.items()}

def get_similar_items_tfidf(item_id, top_n=10):
    """
    Return top_n most similar items to a given item_id using TF-IDF cosine similarity.
    """
    if item_id not in item_to_idx:
        print(f'Item {item_id} not found in catalog.')
        return pd.DataFrame()

    idx = item_to_idx[item_id]
    item_vec = tfidf_matrix[idx]  # sparse vector for this item

    # Compute cosine similarity against all items
    scores = cosine_similarity(item_vec, tfidf_matrix).flatten()
    scores[idx] = 0  # exclude the item itself

    # Get top_n indices
    top_indices = np.argsort(scores)[::-1][:top_n]

    results = catalog.iloc[top_indices][['item_id', 'item_title', 'avg_price', 'popularity']].copy()
    results['similarity_score'] = scores[top_indices].round(4)
    return results.reset_index(drop=True)


# Test it — pick any item_id from the catalog
sample_item = catalog.iloc[0]['item_id']
sample_title = catalog.iloc[0]['item_title']
print(f'Finding items similar to: "{sample_title}" ({sample_item})')
print()
get_similar_items_tfidf(sample_item, top_n=10)

## 3. Sentence-BERT Embeddings
Slower to compute but semantically aware — understands that "red mug" and "crimson cup" are similar.

> **Note:** First run downloads ~90MB model. Subsequent runs use cache.

In [ ]:
from sentence_transformers import SentenceTransformer
import os

EMBEDDINGS_PATH = '../data/processed/item_embeddings.npy'

# Load model (downloads once, cached after)
print('Loading Sentence-BERT model...')
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')  # fast + good quality
print('Model loaded.')

# Generate or load cached embeddings
if os.path.exists(EMBEDDINGS_PATH):
    print('Loading cached embeddings...')
    embeddings = np.load(EMBEDDINGS_PATH)
else:
    print(f'Generating embeddings for {len(catalog):,} items...')
    embeddings = sbert_model.encode(
        catalog['item_title'].tolist(),
        batch_size=256,
        show_progress_bar=True,
        convert_to_numpy=True
    )
    np.save(EMBEDDINGS_PATH, embeddings)
    print('Embeddings saved.')

print(f'Embedding matrix shape: {embeddings.shape}')  # (n_items, 384)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

def get_similar_items_sbert(item_id, top_n=10):
    """
    Return top_n most similar items using Sentence-BERT embedding similarity.
    """
    if item_id not in item_to_idx:
        print(f'Item {item_id} not found in catalog.')
        return pd.DataFrame()

    idx = item_to_idx[item_id]
    item_vec = embeddings[idx].reshape(1, -1)

    scores = cos_sim(item_vec, embeddings).flatten()
    scores[idx] = 0  # exclude self

    top_indices = np.argsort(scores)[::-1][:top_n]

    results = catalog.iloc[top_indices][['item_id', 'item_title', 'avg_price', 'popularity']].copy()
    results['similarity_score'] = scores[top_indices].round(4)
    return results.reset_index(drop=True)


# Compare TF-IDF vs SBERT on the same item
print(f'=== TF-IDF results for: "{sample_title}" ===')
display(get_similar_items_tfidf(sample_item, top_n=5))

print(f'\n=== SBERT results for: "{sample_title}" ===')
display(get_similar_items_sbert(sample_item, top_n=5))

## 4. User-Level Content Recommendations
Given a user's interaction history, recommend items similar to what they've engaged with most.

In [ ]:
def get_content_recommendations(user_id, train_df, top_n=10, method='sbert'):
    """
    Recommend items for a user based on their interaction history.

    Strategy:
      1. Get the user's top interacted items (weighted by interaction_weight)
      2. Find items similar to each of those
      3. Aggregate scores, excluding items the user already interacted with
    """
    user_history = train_df[train_df['user_id'] == user_id]

    if user_history.empty:
        print(f'No history for user {user_id} — cold start. Returning popular items.')
        return catalog.nlargest(top_n, 'popularity')[['item_id', 'item_title', 'avg_price', 'popularity']]

    # Aggregate: if user interacted with same item multiple times, sum weights
    user_items = (
        user_history.groupby('item_id')['interaction_weight']
                    .sum()
                    .nlargest(10)  # focus on top 10 most engaged items
    )

    already_seen = set(user_history['item_id'].unique())

    # Accumulate similarity scores across seed items
    score_map = {}
    sim_fn = get_similar_items_sbert if method == 'sbert' else get_similar_items_tfidf

    for item_id, weight in user_items.items():
        similar = sim_fn(item_id, top_n=20)
        for _, row in similar.iterrows():
            if row['item_id'] not in already_seen:
                # Weight similarity by how strongly user engaged with seed item
                score_map[row['item_id']] = score_map.get(row['item_id'], 0) + (
                    row['similarity_score'] * weight
                )

    if not score_map:
        return catalog.nlargest(top_n, 'popularity')[['item_id', 'item_title', 'avg_price', 'popularity']]

    # Build results dataframe
    recs = pd.DataFrame([
        {'item_id': iid, 'content_score': score}
        for iid, score in score_map.items()
    ]).sort_values('content_score', ascending=False).head(top_n)

    recs = recs.merge(catalog[['item_id', 'item_title', 'avg_price', 'popularity']], on='item_id')
    recs['content_score'] = recs['content_score'].round(4)
    return recs.reset_index(drop=True)


# Test on a real user
sample_user = train['user_id'].value_counts().index[5]  # a reasonably active user
print(f'Recommendations for user: {sample_user}')
print(f'Their top interactions:')
display(train[train['user_id'] == sample_user]
        .groupby('item_title')['interaction_weight'].sum()
        .nlargest(5)
        .reset_index())

print('\nContent-based recommendations (SBERT):')
display(get_content_recommendations(sample_user, train, top_n=10, method='sbert'))

## 5. Save Content Scores for Hybrid Model

In [ ]:
import pickle

# Save everything the hybrid model will need from this notebook
content_data = {
    'tfidf_matrix':  tfidf_matrix,
    'tfidf_model':   tfidf,
    'sbert_embeddings': embeddings,
    'item_to_idx':   item_to_idx,
    'idx_to_item':   idx_to_item,
}

with open('../data/processed/content_data.pkl', 'wb') as f:
    pickle.dump(content_data, f)

print('Saved content_data.pkl')
print('Run notebook 03 next for collaborative filtering.')